In [11]:
!pip install ultralytics -q

In [12]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [13]:
import urllib.request
import ssl
import zipfile
import os

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
print("Downloading Construction-PPE dataset (~178 MB)...")

req = urllib.request.Request(url)
with urllib.request.urlopen(req, context=ctx) as response:
    with open("ppe.zip", "wb") as out_file:
        out_file.write(response.read())
print("Download complete.")

# Extract
with zipfile.ZipFile("ppe.zip", "r") as z:
    z.extractall(".")
print("Extracted.")

# Debug: list top-level items
print("Top-level items after extraction:")
for item in sorted(os.listdir(".")):
    if os.path.isdir(item):
        print(f"  [dir]  {item}")
    else:
        print(f"  [file] {item}")

# Check what's inside images/ and labels/
for d in ["images", "labels"]:
    if os.path.exists(d):
        print(f"\nContents of {d}/:")
        for item in sorted(os.listdir(d)):
            print(f"  {item}")

# Read the data.yaml that came with the dataset
if os.path.exists("data.yaml"):
    print("\ndata.yaml content:")
    with open("data.yaml") as f:
        for line in f:
            print(f"  {line.rstrip()}")

# Determine structure
# If train/valid/test folders exist inside images/labels, use those directly
# Otherwise use the original structure from a subfolder
if os.path.exists("images/train"):
    extracted = "."
    print("\nStructure: images/{train,val,test}/ and labels/{train,val,test}/")
else:
    # Fallback: look for nested structure
    extracted = None
    for item in os.listdir("."):
        if os.path.isdir(item) and item not in ["sample_data", ".config", "images", "labels"]:
            if os.path.exists(os.path.join(item, "train", "images")):
                extracted = item
                break
    print(f"\nDataset root found at: '{extracted}'")

    # If still not found, the dataset might be flat
    if extracted is None and os.path.exists("images") and os.path.exists("labels"):
        # Check if images/ and labels/ contain train/val/test subdirs
        has_splits = all(os.path.exists(f"images/{s}") for s in ["train", "val", "test"])
        if has_splits:
            extracted = "."
            print("Using flat structure with train/val/test subfolders")
        else:
            print("ERROR: Cannot determine dataset structure")

Download complete.
Extracted.
Top-level items after extraction:
  [dir]  .config
  [file] LICENSE
  [file] data.yaml
  [dir]  glove_data
  [dir]  images
  [dir]  labels
  [file] ppe.zip
  [dir]  runs
  [dir]  sample_data
  [file] yolov8s.pt

Contents of images/:
  test
  train
  val

Contents of labels/:
  test
  train
  val

data.yaml content:
  # Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
  
  # Construction-PPE dataset by Ultralytics
  # Documentation: https://docs.ultralytics.com/datasets/detect/construction-ppe/
  # Example usage: yolo train data=construction-ppe.yaml
  # parent
  # ├── ultralytics
  # └── datasets
  #     └── construction-ppe ← downloads here (178.4 MB)
  
  # Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
  path: construction-ppe # dataset root dir
  train: images/train # train images (relative to 'path') 1132 images
  val: images/val # val images (relative to 'path') 1

In [14]:
from pathlib import Path
import shutil

dst = Path("glove_data")

# Detect source structure
def get_source_paths(base, split_name):
    """Find image and label source directories for a split."""
    candidates = []

    # Try: images/train/ and labels/train/
    if (base / "images" / split_name).exists():
        candidates.append((base / "images" / split_name, base / "labels" / split_name))

    # Try: train/images/ and train/labels/
    if (base / split_name / "images").exists():
        candidates.append((base / split_name / "images", base / split_name / "labels"))

    # Try: val -> valid mapping
    alt_split = "valid" if split_name == "val" else split_name
    if (base / "images" / alt_split).exists():
        candidates.append((base / "images" / alt_split, base / "labels" / alt_split))
    if (base / alt_split / "images").exists():
        candidates.append((base / alt_split / "images", base / alt_split / "labels"))

    return candidates[0] if candidates else (None, None)

src = Path(extracted) if extracted else Path(".")

for split in ["train", "val", "test"]:
    (dst / split / "images").mkdir(parents=True, exist_ok=True)
    (dst / split / "labels").mkdir(parents=True, exist_ok=True)

    img_src, lbl_src = get_source_paths(src, split)

    if img_src is None or not img_src.exists():
        print(f"WARNING: Could not find images for {split} split")
        continue

    # Copy images
    for img in img_src.glob("*"):
        shutil.copy(img, dst / split / "images" / img.name)

    # Filter labels: keep class 1 (gloves -> 0) and class 9 (no_gloves -> 1)
    if lbl_src and lbl_src.exists():
        for lbl in lbl_src.glob("*.txt"):
            new_lines = []
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls = int(parts[0])
                        if cls == 1:
                            new_lines.append(f"0 {' '.join(parts[1:])}")
                        elif cls == 9:
                            new_lines.append(f"1 {' '.join(parts[1:])}")
            if new_lines:
                with open(dst / split / "labels" / lbl.name, "w") as f:
                    f.write("\n".join(new_lines) + "\n")

print("Dataset filtered and saved to glove_data/")


Dataset filtered and saved to glove_data/


In [15]:
import yaml

cfg = {
    "path": "/content/glove_data",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 2,
    "names": ["gloved_hand", "bare_hand"]
}

with open("glove_data/data.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("data.yaml created:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

data.yaml created:
  path: /content/glove_data
  train: train/images
  val: val/images
  test: test/images
  nc: 2
  names: ['gloved_hand', 'bare_hand']


In [16]:
from ultralytics import YOLO

EPOCHS = 100
IMG_SIZE = 800
BATCH = 16
CLS_WEIGHT = 2.0

model = YOLO("yolov8s.pt")
model.train(
    data="glove_data/data.yaml",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    cls=CLS_WEIGHT,
    cos_lr=True,
    patience=25,
    mixup=0.2,
    name="glove-detection-final",
    exist_ok=True,
    verbose=True,
)


New https://pypi.org/project/ultralytics/8.4.55 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=glove_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=glove-detec

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7af6c9a16900>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [17]:
from google.colab import files
files.download("runs/detect/glove-detection-final/weights/best.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>